In [2]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import MessagePassing

/Users/arulnanda/Desktop/polymap/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [3]:
class TemporalMemory(nn.Module):

    def __init__(
        self,
        node_dim,
        edge_dim,
        hidden_dim=32
    ):
        super().__init__()

        self.message_mlp = nn.Sequential(
            nn.Linear(
                node_dim * 2 + edge_dim,
                hidden_dim
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                hidden_dim
            )
        )

        self.memory_update = nn.GRUCell(
            hidden_dim,
            hidden_dim
        )

    def forward(
        self,
        memory,
        source,
        target,
        edge_attr
    ):

        source_memory = memory[source]
        target_memory = memory[target]

        message_input = torch.cat(
            [
                source_memory,
                target_memory,
                edge_attr
            ],
            dim=1
        )

        message = self.message_mlp(
            message_input
        )

        new_source_memory = self.memory_update(
            message,
            source_memory
        )

        new_target_memory = self.memory_update(
            message,
            target_memory
        )

        memory = memory.clone()

        memory[source] = new_source_memory
        memory[target] = new_target_memory

        return memory

In [4]:
temporal_edges = pd.read_parquet(
    "../data/processed/temporal_edges.parquet"
)

wallet_features = pd.read_parquet(
    "../data/processed/wallet_features.parquet"
)

print("Temporal edges:", temporal_edges.shape)
print("Wallet features:", wallet_features.shape)

Temporal edges: (103779, 17)
Wallet features: (2356, 15)


In [5]:
from polymap.graph import build_graph

data, wallet_to_id, id_to_wallet, node_features_graph, scaler = (
    build_graph(
        temporal_edges,
        wallet_features
    )
)

print(data)
print("Number of wallets:", data.x.shape[0])
print("Number of temporal interactions:", data.edge_index.shape[1])

Data(x=[2356, 9], edge_index=[2, 103779], edge_attr=[103779, 4], timestamp=[103779])
Number of wallets: 2356
Number of temporal interactions: 103779


In [6]:
sort_idx = torch.argsort(data.timestamp)

edge_index = data.edge_index[:, sort_idx]
edge_attr = data.edge_attr[sort_idx]
timestamps = data.timestamp[sort_idx]

print("First timestamp:", timestamps[0])
print("Last timestamp:", timestamps[-1])

First timestamp: tensor(1784511709000)
Last timestamp: tensor(1790013237000)


In [7]:
timestamps_seconds = timestamps.float()

timestamps_relative = (
    timestamps_seconds - timestamps_seconds[0]
)

print(
    "Time span:",
    timestamps_relative[-1].item() / 3600,
    "hours"
)

Time span: 1528190.2933333332 hours


In [8]:
x = data.x

num_nodes = x.shape[0]

print("Number of nodes:", num_nodes)
print("Node feature dimension:", x.shape[1])

Number of nodes: 2356
Node feature dimension: 9


In [9]:
hidden_dim = 32

node_encoder = nn.Linear(
    x.shape[1],
    hidden_dim
)

In [10]:
initial_memory = node_encoder(x)

print(initial_memory.shape)

torch.Size([2356, 32])


[A memory | B memory | edge features | Δt]
                         ↓
                    message MLP
                         ↓
                      message
                         ↓
                 GRU memory update
                    ↙         ↘
             A new memory   B new memory

In [11]:
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

# Move everything to the same device
x = x.to(device)
edge_index = edge_index.to(device)
edge_attr = edge_attr.to(device)
timestamps_relative = timestamps_relative.to(device)

# Move the encoder BEFORE using it
node_encoder = node_encoder.to(device)

# Create initial wallet memories
initial_memory = node_encoder(x)

print("Initial memory shape:", initial_memory.shape)

Device: mps
Initial memory shape: torch.Size([2356, 32])


In [12]:
x = x.to(device)
edge_index = edge_index.to(device)
edge_attr = edge_attr.to(device)
timestamps_relative = timestamps_relative.to(device)

initial_memory = node_encoder(
    x
).to(device)

temporal_model = TemporalMemory(
    node_dim=hidden_dim,
    edge_dim=edge_attr.shape[1],
    hidden_dim=hidden_dim
).to(device)

In [13]:
temporal_edge_attr = edge_attr.clone()

# Convert time difference from seconds to minutes
temporal_edge_attr[:, 0] = (
    temporal_edge_attr[:, 0] / 60.0
)

print(temporal_edge_attr[:5])

tensor([[0.0000, 1.0000, 0.0000, 1.0000],
        [0.0000, 1.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 1.0000, 0.6609],
        [0.0000, 0.0000, 1.0000, 0.3391],
        [0.0000, 1.0000, 1.0000, 0.5131]], device='mps:0')


In [14]:
memory = initial_memory.detach().clone()

source = edge_index[0, 0].item()
target = edge_index[1, 0].item()

edge = temporal_edge_attr[0].unsqueeze(0)

memory_test = temporal_model(
    memory,
    torch.tensor([source], device=device),
    torch.tensor([target], device=device),
    edge
)

print("Original memory shape:", memory.shape)
print("Updated memory shape:", memory_test.shape)

Original memory shape: torch.Size([2356, 32])
Updated memory shape: torch.Size([2356, 32])


In [15]:
time_diff_mean = temporal_edge_attr[:, 0].mean()
time_diff_std = temporal_edge_attr[:, 0].std()

temporal_edge_attr[:, 0] = (
    temporal_edge_attr[:, 0] - time_diff_mean
) / (time_diff_std + 1e-8)

print(
    "Time difference mean:",
    time_diff_mean.item()
)

print(
    "Time difference std:",
    time_diff_std.item()
)

Time difference mean: 0.6405432224273682
Time difference std: 1.2504150867462158


In [16]:
print(temporal_edge_attr.shape)
print(temporal_edge_attr[:5])

torch.Size([103779, 4])
tensor([[-0.5123,  1.0000,  0.0000,  1.0000],
        [-0.5123,  1.0000,  0.0000,  1.0000],
        [-0.5123,  0.0000,  1.0000,  0.6609],
        [-0.5123,  0.0000,  1.0000,  0.3391],
        [-0.5123,  1.0000,  1.0000,  0.5131]], device='mps:0')


In [17]:
temporal_model = TemporalMemory(
    node_dim=hidden_dim,
    edge_dim=4,
    hidden_dim=hidden_dim
).to(device)

In [18]:
memory = initial_memory.clone()

source = edge_index[0, 0].item()
target = edge_index[1, 0].item()

edge = temporal_edge_attr[0].unsqueeze(0)

memory_test = temporal_model(
    memory,
    torch.tensor([source], device=device),
    torch.tensor([target], device=device),
    edge
)

print("Original memory shape:", memory.shape)
print("Updated memory shape:", memory_test.shape)

Original memory shape: torch.Size([2356, 32])
Updated memory shape: torch.Size([2356, 32])


In [19]:
def process_temporal_events(
    model,
    initial_memory,
    edge_index,
    edge_attr
):
    memory = initial_memory.clone()

    for i in range(edge_index.shape[1]):

        source = edge_index[0, i]
        target = edge_index[1, i]

        memory = model(
            memory,
            source.unsqueeze(0),
            target.unsqueeze(0),
            edge_attr[i].unsqueeze(0)
        )

    return memory


In [20]:
import time

start = time.time()

test_memory = process_temporal_events(
    temporal_model,
    initial_memory,
    edge_index[:, :1000],
    temporal_edge_attr[:1000]
)

elapsed = time.time() - start

print("Time for 1,000 events:", elapsed, "seconds")
print("Memory shape:", test_memory.shape)

Time for 1,000 events: 1.053999900817871 seconds
Memory shape: torch.Size([2356, 32])


In [21]:
class TemporalLinkPredictor(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, source_memory, target_memory):

        pair = torch.cat(
            [source_memory, target_memory],
            dim=1
        )

        return self.mlp(pair).squeeze(1)

In [22]:
link_predictor = TemporalLinkPredictor(
    hidden_dim
).to(device)

print(link_predictor)

TemporalLinkPredictor(
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [23]:
optimizer = torch.optim.Adam(
    list(temporal_model.parameters()) +
    list(link_predictor.parameters()),
    lr=0.001,
    weight_decay=1e-5
)

In [24]:
memory = initial_memory.clone()

source = edge_index[0, 0].item()
target = edge_index[1, 0].item()

source_memory = memory[source].unsqueeze(0)
target_memory = memory[target].unsqueeze(0)

score = link_predictor(
    source_memory,
    target_memory
)

print("Score shape:", score.shape)
print("Score:", score.item())

Score shape: torch.Size([1])
Score: 0.012337945401668549


In [25]:
def train_temporal_chunk(
    temporal_model,
    link_predictor,
    memory,
    edge_index,
    edge_attr,
    start_idx,
    num_events,
    optimizer,
    num_nodes,
    device
):
    temporal_model.train()
    link_predictor.train()

    end_idx = min(start_idx + num_events, edge_index.shape[1])

    losses = []

    for i in range(start_idx, end_idx):
        source = edge_index[0, i].unsqueeze(0)
        target = edge_index[1, i].unsqueeze(0)

        source_memory = memory[source]
        target_memory = memory[target]

        positive_score = link_predictor(source_memory, target_memory)

        negative_target = torch.randint(0, num_nodes, (1,), device=device)
        negative_memory = memory[negative_target]
        negative_score = link_predictor(source_memory, negative_memory)

        positive_loss = F.softplus(-positive_score)
        negative_loss = F.softplus(negative_score)
        losses.append(positive_loss + negative_loss)

        memory = temporal_model(memory, source, target, edge_attr[i].unsqueeze(0))

    loss = torch.stack(losses).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    memory = memory.detach()  # cut graph before next chunk

    return memory, loss.item()   # <-- was missing

In [26]:
def train_temporal_chunk(
    temporal_model,
    link_predictor,
    memory,
    edge_index,
    edge_attr,
    start_idx,
    num_events,
    optimizer,
    num_nodes,
    device
):
    temporal_model.train()
    link_predictor.train()

    end_idx = min(start_idx + num_events, edge_index.shape[1])

    losses = []

    for i in range(start_idx, end_idx):
        source = edge_index[0, i].unsqueeze(0)
        target = edge_index[1, i].unsqueeze(0)

        source_memory = memory[source]
        target_memory = memory[target]

        positive_score = link_predictor(source_memory, target_memory)

        negative_target = torch.randint(0, num_nodes, (1,), device=device)
        negative_memory = memory[negative_target]
        negative_score = link_predictor(source_memory, negative_memory)

        positive_loss = F.softplus(-positive_score)
        negative_loss = F.softplus(negative_score)
        losses.append(positive_loss + negative_loss)

        memory = temporal_model(memory, source, target, edge_attr[i].unsqueeze(0))

    loss = torch.stack(losses).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    memory = memory.detach()  # cut graph before next chunk

    return memory, loss.item()   # <-- was missing

In [27]:
memory = initial_memory.detach().clone()

start_idx = 0
chunk_size = 2000  # tune based on your per-event timing

while start_idx < edge_index.shape[1]:
    memory, loss = train_temporal_chunk(
        temporal_model=temporal_model,
        link_predictor=link_predictor,
        memory=memory,
        edge_index=edge_index,
        edge_attr=temporal_edge_attr,
        start_idx=start_idx,
        num_events=chunk_size,
        optimizer=optimizer,
        num_nodes=num_nodes,
        device=device
    )
    print(f"Events {start_idx}-{start_idx+chunk_size}: loss={loss:.4f}")
    start_idx += chunk_size

Events 0-2000: loss=1.3962
Events 2000-4000: loss=1.3886
Events 4000-6000: loss=1.3749
Events 6000-8000: loss=1.3625
Events 8000-10000: loss=1.3603
Events 10000-12000: loss=1.3601
Events 12000-14000: loss=1.3525
Events 14000-16000: loss=1.3466
Events 16000-18000: loss=1.3463
Events 18000-20000: loss=1.3343
Events 20000-22000: loss=1.3301
Events 22000-24000: loss=1.3168
Events 24000-26000: loss=1.3101
Events 26000-28000: loss=1.2949
Events 28000-30000: loss=1.2749
Events 30000-32000: loss=1.2711
Events 32000-34000: loss=1.2265
Events 34000-36000: loss=1.2063
Events 36000-38000: loss=1.1832
Events 38000-40000: loss=1.1990
Events 40000-42000: loss=1.1507
Events 42000-44000: loss=1.1052
Events 44000-46000: loss=1.0947
Events 46000-48000: loss=1.0985
Events 48000-50000: loss=1.0688
Events 50000-52000: loss=1.0510
Events 52000-54000: loss=1.0313
Events 54000-56000: loss=1.0007
Events 56000-58000: loss=0.9519
Events 58000-60000: loss=0.9237
Events 60000-62000: loss=0.8954
Events 62000-64000: 

In [28]:
# Reset temporal model
temporal_model = TemporalMemory(
    node_dim=hidden_dim,
    edge_dim=4,
    hidden_dim=hidden_dim
).to(device)

# Reset link predictor
link_predictor = TemporalLinkPredictor(
    hidden_dim
).to(device)

# New optimizer
optimizer = torch.optim.Adam(
    list(temporal_model.parameters()) +
    list(link_predictor.parameters()),
    lr=0.001,
    weight_decay=1e-5
)

# Fresh initial memory
memory = initial_memory.detach().clone()

print("Model reset.")
print("Memory shape:", memory.shape)

Model reset.
Memory shape: torch.Size([2356, 32])


In [29]:
memory.shape

torch.Size([2356, 32])

In [30]:
tgnn_embeddings = (
    memory.detach()
    .cpu()
    .numpy()
)

print("TGNN embeddings:", tgnn_embeddings.shape)

TGNN embeddings: (2356, 32)


In [31]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

tgnn_results = {}

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(
        tgnn_embeddings
    )

    silhouette = silhouette_score(
        tgnn_embeddings,
        labels
    )

    tgnn_results[k] = silhouette

    print(
        f"K={k}: "
        f"Silhouette={silhouette:.4f}"
    )

K=2: Silhouette=0.7232
K=3: Silhouette=0.3073
K=4: Silhouette=0.3703
K=5: Silhouette=0.3788
K=6: Silhouette=0.3940
K=7: Silhouette=0.4302
K=8: Silhouette=0.4739
K=9: Silhouette=0.4907
K=10: Silhouette=0.5038


In [32]:
kmeans_tgnn = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

tgnn_labels = kmeans_tgnn.fit_predict(
    tgnn_embeddings
)

tgnn_clustered = wallet_features.copy()

tgnn_clustered["tgnn_cluster"] = tgnn_labels

cluster_summary = (
    tgnn_clustered
    .groupby("tgnn_cluster")
    .agg(
        wallets=("proxyWallet", "size"),
        mean_trades=("trade_count", "mean"),
        median_trades=("trade_count", "median"),
        mean_trade_size=("avg_trade_size", "mean"),
        median_trade_size=("median_trade_size", "median"),
        avg_price=("avg_price", "mean"),
        buy_ratio=("buy_ratio", "mean"),
        sell_ratio=("sell_ratio", "mean"),
        mean_duration=("active_duration_hours", "mean")
    )
)

cluster_summary

,wallets,mean_trades,median_trades,mean_trade_size,median_trade_size,avg_price,buy_ratio,sell_ratio,mean_duration
tgnn_cluster,,,,,,,,,
0,2292,3.102967,1.0,721.384108,20.32505,0.714621,0.493543,0.506457,122.998086
1,64,45.125000,9.0,396.269322,36.32000,0.572399,0.613609,0.386391,674.890877


In [33]:
from scipy.stats import pearsonr

embedding_norm = np.linalg.norm(
    tgnn_embeddings,
    axis=1
)

print(
    "Embedding norm ↔ trade count:",
    pearsonr(
        embedding_norm,
        wallet_features["trade_count"]
    )[0]
)

print(
    "Embedding norm ↔ active duration:",
    pearsonr(
        embedding_norm,
        wallet_features["active_duration_hours"]
    )[0]
)

print(
    "Embedding norm ↔ avg trade size:",
    pearsonr(
        embedding_norm,
        wallet_features["avg_trade_size"]
    )[0]
)

Embedding norm ↔ trade count: 0.3823788830392235
Embedding norm ↔ active duration: 0.47263230401141376
Embedding norm ↔ avg trade size: 0.09648521805068361


In [34]:
# Chronological train/test split

num_events = len(temporal_edges)

split_idx = int(num_events * 0.70)

train_edges = temporal_edges.iloc[:split_idx].copy()
test_edges = temporal_edges.iloc[split_idx:].copy()

print("Total events:", num_events)
print("Training events:", len(train_edges))
print("Test events:", len(test_edges))

print("\nTraining period:")
print(train_edges["timestamp"].min())
print(train_edges["timestamp"].max())

print("\nTest period:")
print(test_edges["timestamp"].min())
print(test_edges["timestamp"].max())

Total events: 103779
Training events: 72645
Test events: 31134

Training period:
2026-07-20 01:41:49+00:00
2026-09-13 15:46:36+00:00

Test period:
2026-09-13 15:42:15+00:00
2026-09-21 17:53:57+00:00


In [35]:
# Sort all temporal interactions chronologically
temporal_edges = temporal_edges.sort_values(
    "timestamp"
).reset_index(drop=True)

# Now split chronologically
num_events = len(temporal_edges)
split_idx = int(num_events * 0.70)

train_edges = temporal_edges.iloc[:split_idx].copy()
test_edges = temporal_edges.iloc[split_idx:].copy()

print("Total events:", num_events)
print("Training events:", len(train_edges))
print("Test events:", len(test_edges))

print("\nTraining period:")
print(train_edges["timestamp"].min())
print(train_edges["timestamp"].max())

print("\nTest period:")
print(test_edges["timestamp"].min())
print(test_edges["timestamp"].max())

Total events: 103779
Training events: 72645
Test events: 31134

Training period:
2026-07-20 01:41:49+00:00
2026-09-13 15:42:42+00:00

Test period:
2026-09-13 15:42:42+00:00
2026-09-21 17:53:57+00:00


In [36]:
assert (
    train_edges["timestamp"].max()
    <=
    test_edges["timestamp"].min()
)

print("✓ Valid chronological split")

✓ Valid chronological split


In [37]:
# Build wallet features using TRAINING DATA ONLY

train_wallet_features = (
    train_trades_df
    .groupby("proxyWallet")
    .agg(
        trade_count=("proxyWallet", "size"),
        buy_count=("side", lambda x: (x == "BUY").sum()),
        sell_count=("side", lambda x: (x == "SELL").sum()),
        avg_trade_size=("size", "mean"),
        median_trade_size=("size", "median"),
        avg_price=("price", "mean"),
        price_std=("price", "std"),
        unique_outcomes=("outcome", "nunique"),
        first_trade=("datetime", "min"),
        last_trade=("datetime", "max")
    )
    .reset_index()
)

train_wallet_features["buy_ratio"] = (
    train_wallet_features["buy_count"] /
    train_wallet_features["trade_count"]
)

train_wallet_features["sell_ratio"] = (
    train_wallet_features["sell_count"] /
    train_wallet_features["trade_count"]
)

train_wallet_features["active_duration_hours"] = (
    (
        train_wallet_features["last_trade"] -
        train_wallet_features["first_trade"]
    ).dt.total_seconds() / 3600
)

print("Training wallets:", len(train_wallet_features))
train_wallet_features.head()

NameError: name 'train_trades_df' is not defined

In [38]:
import os

print(os.listdir("../data/processed"))

['temporal_edges.parquet', 'baseline_clusters.parquet', 'wallet_features.parquet']


In [39]:
temporal_edges.columns.tolist()

['source',
 'target',
 'timestamp',
 'time_diff',
 'same_side',
 'same_outcome',
 'price_diff',
 'size_ratio',
 'source_side',
 'target_side',
 'source_outcome',
 'target_outcome',
 'source_size',
 'target_size',
 'source_price',
 'target_price',
 'same_behavior']

In [40]:
print("Temporal edges:", len(temporal_edges))
print("Unique source wallets:", temporal_edges["source"].nunique())
print("Unique target wallets:", temporal_edges["target"].nunique())

all_graph_wallets = set(temporal_edges["source"]) | set(temporal_edges["target"])

print("Unique graph wallets:", len(all_graph_wallets))

Temporal edges: 103779
Unique source wallets: 2051
Unique target wallets: 1780
Unique graph wallets: 2356


In [41]:
import requests
import pandas as pd
import time

GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"

market_id = "665374"

response = requests.get(
    f"{GAMMA_API}/markets/{market_id}",
    timeout=30
)

response.raise_for_status()

market = response.json()

condition_id = market["conditionId"]

print("Market:", market["question"])
print("Condition ID:", condition_id)

Market: Will the U.S. invade Iran before 2027?
Condition ID: 0x5db999fad322cea2914535aae5517060c3f80ad6d8c0231cde2124a434d16846


In [42]:
all_trades = []

for offset in range(0, 10000, 100):

    response = requests.get(
        f"{DATA_API}/trades",
        params={
            "market": condition_id,
            "limit": 100,
            "offset": offset,
            "takerOnly": False
        },
        timeout=30
    )

    response.raise_for_status()

    batch = response.json()

    if not batch:
        break

    all_trades.extend(batch)

    print(
        f"Offset {offset}: {len(batch)} trades"
    )

    if len(batch) < 100:
        break

    time.sleep(0.2)

raw_trades = pd.DataFrame(all_trades)

print("\nTotal trades:", len(raw_trades))
print("Unique wallets:", raw_trades["proxyWallet"].nunique())

Offset 0: 100 trades
Offset 100: 100 trades
Offset 200: 100 trades
Offset 300: 100 trades
Offset 400: 100 trades
Offset 500: 100 trades
Offset 600: 100 trades
Offset 700: 100 trades
Offset 800: 100 trades
Offset 900: 100 trades
Offset 1000: 100 trades
Offset 1100: 100 trades
Offset 1200: 100 trades
Offset 1300: 100 trades
Offset 1400: 100 trades
Offset 1500: 100 trades
Offset 1600: 100 trades
Offset 1700: 100 trades
Offset 1800: 100 trades
Offset 1900: 100 trades
Offset 2000: 100 trades
Offset 2100: 100 trades
Offset 2200: 100 trades
Offset 2300: 100 trades
Offset 2400: 100 trades
Offset 2500: 100 trades
Offset 2600: 100 trades
Offset 2700: 100 trades
Offset 2800: 100 trades
Offset 2900: 100 trades
Offset 3000: 100 trades
Offset 3100: 100 trades
Offset 3200: 100 trades
Offset 3300: 100 trades
Offset 3400: 100 trades
Offset 3500: 100 trades
Offset 3600: 100 trades
Offset 3700: 100 trades
Offset 3800: 100 trades
Offset 3900: 100 trades
Offset 4000: 100 trades
Offset 4100: 100 trades
Offs

In [43]:
raw_trades["datetime"] = pd.to_datetime(
    raw_trades["timestamp"],
    unit="s",
    utc=True
)

raw_trades = (
    raw_trades
    .sort_values("datetime")
    .reset_index(drop=True)
)

In [44]:
raw_trades["datetime"] = pd.to_datetime(
    raw_trades["timestamp"],
    unit="s",
    utc=True
)

In [45]:
print("RAW TRADES")
print("Min:", raw_trades["datetime"].min())
print("Max:", raw_trades["datetime"].max())

print("\nTEMPORAL EDGES")
print("Min:", temporal_edges["timestamp"].min())
print("Max:", temporal_edges["timestamp"].max())

RAW TRADES
Min: 2026-09-10 22:49:07+00:00
Max: 2026-09-26 10:09:32+00:00

TEMPORAL EDGES
Min: 2026-07-20 01:41:49+00:00
Max: 2026-09-21 17:53:57+00:00


In [46]:
raw_wallets = set(raw_trades["proxyWallet"])

edge_wallets = (
    set(temporal_edges["source"])
    |
    set(temporal_edges["target"])
)

print("\nRaw trade wallets:", len(raw_wallets))
print("Graph wallets:", len(edge_wallets))

print("Wallet overlap:", len(raw_wallets & edge_wallets))

print(
    "Graph wallets missing from raw:",
    len(edge_wallets - raw_wallets)
)

print(
    "Raw wallets missing from graph:",
    len(raw_wallets - edge_wallets)
)


Raw trade wallets: 1353
Graph wallets: 2356
Wallet overlap: 177
Graph wallets missing from raw: 2179
Raw wallets missing from graph: 1176


In [47]:
# Ensure chronological order
temporal_edges = (
    temporal_edges
    .sort_values("timestamp")
    .reset_index(drop=True)
)

split_idx = int(len(temporal_edges) * 0.70)

train_edges = temporal_edges.iloc[:split_idx].copy()
test_edges = temporal_edges.iloc[split_idx:].copy()

cutoff_time = train_edges["timestamp"].max()

print("Training events:", len(train_edges))
print("Test events:", len(test_edges))
print("Cutoff:", cutoff_time)

Training events: 72645
Test events: 31134
Cutoff: 2026-09-13 15:42:42+00:00


In [48]:
def extract_wallet_events(edges):
    
    source_events = edges[
        [
            "source",
            "timestamp",
            "source_side",
            "source_outcome",
            "source_size",
            "source_price"
        ]
    ].copy()

    source_events = source_events.rename(
        columns={
            "source": "proxyWallet",
            "source_side": "side",
            "source_outcome": "outcome",
            "source_size": "size",
            "source_price": "price"
        }
    )

    target_events = edges[
        [
            "target",
            "timestamp",
            "target_side",
            "target_outcome",
            "target_size",
            "target_price"
        ]
    ].copy()

    target_events = target_events.rename(
        columns={
            "target": "proxyWallet",
            "target_side": "side",
            "target_outcome": "outcome",
            "target_size": "size",
            "target_price": "price"
        }
    )

    events = pd.concat(
        [source_events, target_events],
        ignore_index=True
    )

    events = events.drop_duplicates(
        subset=[
            "proxyWallet",
            "timestamp",
            "side",
            "outcome",
            "size",
            "price"
        ]
    )

    return events

In [49]:
train_events = extract_wallet_events(train_edges)

print("Reconstructed training events:", len(train_events))
print("Training wallets:", train_events["proxyWallet"].nunique())

train_events.head()

Reconstructed training events: 20630
Training wallets: 2140


,proxyWallet,timestamp,side,outcome,size,price
0,0x2a778bad898897c5eda9028075a80fc4f74e9ca4,2026-07-20 01:41:49+00:00,BUY,No,6.89,0.92
1,0x188fda5454e8862e9e371f83715c30c066192a13,2026-07-20 01:57:13+00:00,BUY,No,7.26,0.92
2,0x0196b30b35a96aad147366a30c2e25f00382f175,2026-07-20 02:05:42+00:00,SELL,No,34.00,0.91
4,0x660622caad009a8cc1f38038c0913be658621338,2026-07-20 02:05:42+00:00,BUY,No,22.47,0.91
5,0x660622caad009a8cc1f38038c0913be658621338,2026-07-20 02:07:55+00:00,BUY,No,22.47,0.91


In [50]:
initial_memory = nn.Parameter(
    torch.zeros(num_nodes, hidden_dim)
)

In [51]:
# Use the same wallet mapping for train and test
wallets = sorted(
    set(temporal_edges["source"])
    .union(temporal_edges["target"])
)

wallet_to_id = {
    wallet: idx
    for idx, wallet in enumerate(wallets)
}

id_to_wallet = {
    idx: wallet
    for wallet, idx in wallet_to_id.items()
}

num_nodes = len(wallet_to_id)

print("Number of wallets:", num_nodes)
print("Train interactions:", len(train_edges))
print("Test interactions:", len(test_edges))

Number of wallets: 2356
Train interactions: 72645
Test interactions: 31134


In [52]:
def edges_to_tensors(edges):
    source = (
        edges["source"]
        .map(wallet_to_id)
        .values
    )

    target = (
        edges["target"]
        .map(wallet_to_id)
        .values
    )

    edge_index = torch.tensor(
        np.vstack([source, target]),
        dtype=torch.long,
        device=device
    )

    edge_attr = torch.tensor(
        edges[
            [
                "time_diff",
                "same_side",
                "same_outcome",
                "size_ratio"
            ]
        ].values.astype(np.float32),
        dtype=torch.float32,
        device=device
    )

    return edge_index, edge_attr

In [53]:
train_time = (
    train_edges["time_diff"]
    .values
    .astype(np.float32)
    / 60.0
)

time_mean = train_time.mean()
time_std = train_time.std()

print("Train time mean:", time_mean)
print("Train time std:", time_std)

Train time mean: 0.76953864
Train time std: 1.3926182


In [54]:
def make_edge_attributes(edges):

    arr = edges[
        [
            "time_diff",
            "same_side",
            "same_outcome",
            "size_ratio"
        ]
    ].values.astype(np.float32)

    # Convert seconds → minutes
    arr[:, 0] = arr[:, 0] / 60.0

    # Normalize using TRAIN statistics
    arr[:, 0] = (
        arr[:, 0] - time_mean
    ) / (time_std + 1e-8)

    return torch.tensor(
        arr,
        dtype=torch.float32,
        device=device
    )

In [55]:
train_edge_index, train_edge_attr = edges_to_tensors(train_edges)
test_edge_index, test_edge_attr = edges_to_tensors(test_edges)

train_edge_attr = make_edge_attributes(train_edges)
test_edge_attr = make_edge_attributes(test_edges)

print(train_edge_index.shape)
print(test_edge_index.shape)

torch.Size([2, 72645])
torch.Size([2, 31134])


In [56]:
hidden_dim = 32

temporal_model = TemporalMemory(
    node_dim=hidden_dim,
    edge_dim=4,
    hidden_dim=hidden_dim
).to(device)

link_predictor = TemporalLinkPredictor(
    hidden_dim
).to(device)

optimizer = torch.optim.Adam(
    list(temporal_model.parameters()) +
    list(link_predictor.parameters()),
    lr=0.001,
    weight_decay=1e-5
)

In [57]:
memory = torch.zeros(
    num_nodes,
    hidden_dim,
    device=device
)

print(memory.shape)

torch.Size([2356, 32])


In [58]:
chunk_size = 2000

train_losses = []

for start in range(
    0,
    train_edge_index.shape[1],
    chunk_size
):

    memory, loss = train_temporal_chunk(
        temporal_model,
        link_predictor,
        memory,
        train_edge_index,
        train_edge_attr,
        start,
        chunk_size,
        optimizer,
        num_nodes,
        device
    )

    train_losses.append(loss)

    print(
        f"Events {start:,}-"
        f"{min(start + chunk_size, train_edge_index.shape[1]):,} "
        f"| Loss: {loss:.4f}"
    )

Events 0-2,000 | Loss: 1.3963
Events 2,000-4,000 | Loss: 1.3884
Events 4,000-6,000 | Loss: 1.3778
Events 6,000-8,000 | Loss: 1.3682
Events 8,000-10,000 | Loss: 1.3699
Events 10,000-12,000 | Loss: 1.3658
Events 12,000-14,000 | Loss: 1.3611
Events 14,000-16,000 | Loss: 1.3561
Events 16,000-18,000 | Loss: 1.3528
Events 18,000-20,000 | Loss: 1.3419
Events 20,000-22,000 | Loss: 1.3378
Events 22,000-24,000 | Loss: 1.3250
Events 24,000-26,000 | Loss: 1.3159
Events 26,000-28,000 | Loss: 1.3030
Events 28,000-30,000 | Loss: 1.2830
Events 30,000-32,000 | Loss: 1.2762
Events 32,000-34,000 | Loss: 1.2392
Events 34,000-36,000 | Loss: 1.2197
Events 36,000-38,000 | Loss: 1.1966
Events 38,000-40,000 | Loss: 1.2080
Events 40,000-42,000 | Loss: 1.1501
Events 42,000-44,000 | Loss: 1.0799
Events 44,000-46,000 | Loss: 1.0569
Events 46,000-48,000 | Loss: 1.0351
Events 48,000-50,000 | Loss: 0.9630
Events 50,000-52,000 | Loss: 0.9325
Events 52,000-54,000 | Loss: 0.8875
Events 54,000-56,000 | Loss: 0.8233
Event

In [59]:
from sklearn.metrics import roc_auc_score, average_precision_score

temporal_model.eval()
link_predictor.eval()

memory_eval = memory.detach().clone()

y_true = []
y_scores = []

train_wallets = set(train_edges["source"]) | set(train_edges["target"])

seen_test_events = 0
cold_start_events = 0

with torch.no_grad():

    for i in range(test_edge_index.shape[1]):

        source = test_edge_index[0, i]
        target = test_edge_index[1, i]

        source_id = source.item()
        target_id = target.item()

        # Track cold-start events
        if (
            id_to_wallet[source_id] not in train_wallets
            or
            id_to_wallet[target_id] not in train_wallets
        ):
            cold_start_events += 1
        else:
            seen_test_events += 1

        # -------------------------
        # Positive: actual future interaction
        # -------------------------

        source_memory = memory_eval[source]
        target_memory = memory_eval[target]

        positive_score = link_predictor(
            source_memory.unsqueeze(0),
            target_memory.unsqueeze(0)
        ).item()

        y_true.append(1)
        y_scores.append(positive_score)

        # -------------------------
        # Negative: random wallet
        # -------------------------

        negative_target = torch.randint(
            0,
            num_nodes,
            (1,),
            device=device
        )

        # Avoid source and true target
        while (
            negative_target.item() == source_id
            or
            negative_target.item() == target_id
        ):
            negative_target = torch.randint(
                0,
                num_nodes,
                (1,),
                device=device
            )

        negative_memory = memory_eval[negative_target]

        negative_score = link_predictor(
            source_memory.unsqueeze(0),
            negative_memory
        ).item()

        y_true.append(0)
        y_scores.append(negative_score)

        # -------------------------
        # Reveal the actual event
        # -------------------------

        memory_eval = temporal_model(
            memory_eval,
            source.unsqueeze(0),
            target.unsqueeze(0),
            test_edge_attr[i].unsqueeze(0)
        )

print("Evaluation complete.")

y_true = np.array(y_true)
y_scores = np.array(y_scores)

roc_auc = roc_auc_score(
    y_true,
    y_scores
)

average_precision = average_precision_score(
    y_true,
    y_scores
)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Average Precision: {average_precision:.4f}")

print()
print("Test events:", len(test_edges))
print("Seen-wallet test events:", seen_test_events)
print("Cold-start test events:", cold_start_events)

Evaluation complete.
ROC-AUC: 0.9632
Average Precision: 0.9317

Test events: 31134
Seen-wallet test events: 22552
Cold-start test events: 8582


In [60]:
from sklearn.metrics import roc_auc_score, average_precision_score

temporal_model.eval()
link_predictor.eval()

memory_eval = memory.detach().clone()

seen_y_true = []
seen_y_scores = []

cold_y_true = []
cold_y_scores = []

train_wallets = set(
    train_edges["source"]
).union(
    set(train_edges["target"])
)

with torch.no_grad():

    for i in range(test_edge_index.shape[1]):

        source = test_edge_index[0, i]
        target = test_edge_index[1, i]

        source_id = source.item()
        target_id = target.item()

        source_wallet = id_to_wallet[source_id]
        target_wallet = id_to_wallet[target_id]

        is_cold_start = (
            source_wallet not in train_wallets
            or target_wallet not in train_wallets
        )

        # -------------------------
        # Positive
        # -------------------------

        source_memory = memory_eval[source]
        target_memory = memory_eval[target]

        positive_score = link_predictor(
            source_memory.unsqueeze(0),
            target_memory.unsqueeze(0)
        ).item()

        # -------------------------
        # Negative
        # -------------------------

        negative_target = torch.randint(
            0,
            num_nodes,
            (1,),
            device=device
        )

        while (
            negative_target.item() == source_id
            or
            negative_target.item() == target_id
        ):
            negative_target = torch.randint(
                0,
                num_nodes,
                (1,),
                device=device
            )

        negative_score = link_predictor(
            source_memory.unsqueeze(0),
            memory_eval[negative_target]
        ).item()

        # -------------------------
        # Store result
        # -------------------------

        if is_cold_start:

            cold_y_true.extend([1, 0])
            cold_y_scores.extend([
                positive_score,
                negative_score
            ])

        else:

            seen_y_true.extend([1, 0])
            seen_y_scores.extend([
                positive_score,
                negative_score
            ])

        # -------------------------
        # Reveal event
        # -------------------------

        memory_eval = temporal_model(
            memory_eval,
            source.unsqueeze(0),
            target.unsqueeze(0),
            test_edge_attr[i].unsqueeze(0)
        )


seen_auc = roc_auc_score(
    seen_y_true,
    seen_y_scores
)

seen_ap = average_precision_score(
    seen_y_true,
    seen_y_scores
)

cold_auc = roc_auc_score(
    cold_y_true,
    cold_y_scores
)

cold_ap = average_precision_score(
    cold_y_true,
    cold_y_scores
)

print("SEEN-WALLET TEST")
print(f"Events: {len(seen_y_true) // 2:,}")
print(f"ROC-AUC: {seen_auc:.4f}")
print(f"Average Precision: {seen_ap:.4f}")

print()

print("COLD-START TEST")
print(f"Events: {len(cold_y_true) // 2:,}")
print(f"ROC-AUC: {cold_auc:.4f}")
print(f"Average Precision: {cold_ap:.4f}")

SEEN-WALLET TEST
Events: 22,552
ROC-AUC: 0.9692
Average Precision: 0.9394

COLD-START TEST
Events: 8,582
ROC-AUC: 0.9502
Average Precision: 0.9180


In [61]:
from collections import defaultdict

# Count historical interactions for each directed wallet pair
pair_counts = (
    train_edges
    .groupby(["source", "target"])
    .size()
    .to_dict()
)

print("Historical wallet pairs:", len(pair_counts))

Historical wallet pairs: 25054


In [62]:
baseline_y_true = []
baseline_y_scores = []

rng = np.random.default_rng(42)

for _, row in test_edges.iterrows():

    source = row["source"]
    target = row["target"]

    # Positive score
    positive_score = pair_counts.get(
        (source, target),
        0
    )

    baseline_y_true.append(1)
    baseline_y_scores.append(positive_score)

    # Random negative wallet
    negative_target = target

    while (
        negative_target == target
        or
        negative_target == source
    ):
        negative_target = rng.choice(
            list(wallet_to_id.keys())
        )

    negative_score = pair_counts.get(
        (source, negative_target),
        0
    )

    baseline_y_true.append(0)
    baseline_y_scores.append(negative_score)

In [63]:
baseline_auc = roc_auc_score(
    baseline_y_true,
    baseline_y_scores
)

baseline_ap = average_precision_score(
    baseline_y_true,
    baseline_y_scores
)

print(f"Historical Frequency Baseline")
print(f"ROC-AUC: {baseline_auc:.4f}")
print(f"Average Precision: {baseline_ap:.4f}")

Historical Frequency Baseline
ROC-AUC: 0.6744
Average Precision: 0.6747


In [64]:
from collections import defaultdict

# Historical neighbors for each source wallet
historical_neighbors = defaultdict(set)

for _, row in train_edges.iterrows():

    historical_neighbors[row["source"]].add(
        row["target"]
    )

# Count eligible test events
eligible_events = 0

for _, row in test_edges.iterrows():

    source = row["source"]
    target = row["target"]

    candidates = (
        historical_neighbors[source]
        - {target, source}
    )

    if len(candidates) > 0:
        eligible_events += 1

print("Eligible hard-negative events:", eligible_events)
print(
    "Percentage of test:",
    eligible_events / len(test_edges) * 100,
    "%"
)

Eligible hard-negative events: 25826
Percentage of test: 82.95111453716194 %


In [65]:
from sklearn.metrics import roc_auc_score, average_precision_score

temporal_model.eval()
link_predictor.eval()

memory_eval = memory.detach().clone()

hard_y_true = []
hard_y_scores = []

hard_frequency_scores = []

rng = np.random.default_rng(42)

evaluated_events = 0

with torch.no_grad():

    for i in range(test_edge_index.shape[1]):

        source = test_edge_index[0, i]
        target = test_edge_index[1, i]

        source_id = source.item()
        target_id = target.item()

        source_wallet = id_to_wallet[source_id]
        target_wallet = id_to_wallet[target_id]

        # --------------------------------
        # Find historical neighbors
        # --------------------------------

        candidates = list(
            historical_neighbors[source_wallet]
            - {
                target_wallet,
                source_wallet
            }
        )

        # Can't create a hard negative
        # for this source
        if len(candidates) == 0:

            # Still reveal the real event
            memory_eval = temporal_model(
                memory_eval,
                source.unsqueeze(0),
                target.unsqueeze(0),
                test_edge_attr[i].unsqueeze(0)
            )

            continue

        # Pick historical neighbor
        negative_wallet = rng.choice(candidates)

        negative_id = wallet_to_id[
            negative_wallet
        ]

        negative_target = torch.tensor(
            [negative_id],
            dtype=torch.long,
            device=device
        )

        # --------------------------------
        # TGNN positive score
        # --------------------------------

        source_memory = memory_eval[source]
        target_memory = memory_eval[target]

        positive_score = link_predictor(
            source_memory.unsqueeze(0),
            target_memory.unsqueeze(0)
        ).item()

        # --------------------------------
        # TGNN hard-negative score
        # --------------------------------

        negative_memory = memory_eval[
            negative_target
        ]

        negative_score = link_predictor(
            source_memory.unsqueeze(0),
            negative_memory
        ).item()

        hard_y_true.extend([1, 0])
        hard_y_scores.extend([
            positive_score,
            negative_score
        ])

        # --------------------------------
        # Historical-frequency baseline
        # --------------------------------

        positive_frequency = pair_counts.get(
            (source_wallet, target_wallet),
            0
        )

        negative_frequency = pair_counts.get(
            (source_wallet, negative_wallet),
            0
        )

        hard_frequency_scores.extend([
            positive_frequency,
            negative_frequency
        ])

        evaluated_events += 1

        # --------------------------------
        # Reveal actual event
        # --------------------------------

        memory_eval = temporal_model(
            memory_eval,
            source.unsqueeze(0),
            target.unsqueeze(0),
            test_edge_attr[i].unsqueeze(0)
        )


# --------------------------------
# Metrics
# --------------------------------

tgnn_hard_auc = roc_auc_score(
    hard_y_true,
    hard_y_scores
)

tgnn_hard_ap = average_precision_score(
    hard_y_true,
    hard_y_scores
)

frequency_hard_auc = roc_auc_score(
    hard_y_true,
    hard_frequency_scores
)

frequency_hard_ap = average_precision_score(
    hard_y_true,
    hard_frequency_scores
)

print("Hard-negative evaluation")
print("------------------------")
print("Events evaluated:", evaluated_events)

print()
print("TGNN")
print(f"ROC-AUC: {tgnn_hard_auc:.4f}")
print(f"Average Precision: {tgnn_hard_ap:.4f}")

print()
print("Historical Frequency")
print(f"ROC-AUC: {frequency_hard_auc:.4f}")
print(f"Average Precision: {frequency_hard_ap:.4f}")

Hard-negative evaluation
------------------------
Events evaluated: 25826

TGNN
ROC-AUC: 0.7686
Average Precision: 0.6670

Historical Frequency
ROC-AUC: 0.3454
Average Precision: 0.5330


In [66]:
# Historical statistics for each directed wallet pair

pair_stats = (
    train_edges
    .groupby(["source", "target"])
    .agg(
        interaction_count=("target", "size"),
        last_interaction=("timestamp", "max")
    )
    .reset_index()
)

pair_stats.head()

,source,target,interaction_count,last_interaction
0,0x00039cdf685d036595ed8f079a897de78f75e331,0x256c8ad6a66e545a74e50a462df0b94c78bde7d1,1,2026-07-20 15:36:43+00:00
1,0x00039cdf685d036595ed8f079a897de78f75e331,0xa58d4f278d7953cd38eeb929f7e242bfc7c0b9b8,1,2026-07-20 15:36:43+00:00
2,0x00039cdf685d036595ed8f079a897de78f75e331,0xcc2982e3fbbcaf9af9fa373fa3ae392fd9457774,1,2026-07-20 15:36:18+00:00
3,0x0072e290392e1165247c8c2ade58f043fe8a191e,0xd7eee5e918dc59fa32386a4ad8a89655946e9597,2,2026-07-21 00:40:46+00:00
4,0x0072e290392e1165247c8c2ade58f043fe8a191e,0xdd6bb8cc75b2a40ef1a1230b0546fcfb467fb5de,1,2026-07-21 00:40:46+00:00


In [67]:
# Fast lookup:
pair_stats_lookup = {
    (row["source"], row["target"]): {
        "count": row["interaction_count"],
        "last_time": row["last_interaction"]
    }
    for _, row in pair_stats.iterrows()
}

# Historical neighbors
historical_neighbors = defaultdict(set)

for _, row in pair_stats.iterrows():

    historical_neighbors[
        row["source"]
    ].add(
        row["target"]
    )

print(
    "Historical directed pairs:",
    len(pair_stats_lookup)
)

Historical directed pairs: 25054


In [68]:
def choose_matched_negative(
    source,
    positive_target,
    test_time,
    max_candidates=50
):

    candidates = []

    positive_stats = pair_stats_lookup.get(
        (source, positive_target)
    )

    if positive_stats is None:
        return None

    positive_count = positive_stats["count"]

    for candidate in historical_neighbors[source]:

        if candidate == positive_target:
            continue

        stats = pair_stats_lookup[
            (source, candidate)
        ]

        count_difference = abs(
            stats["count"] - positive_count
        )

        time_difference = abs(
            (
                test_time -
                stats["last_time"]
            ).total_seconds()
        )

        candidates.append(
            (
                count_difference,
                time_difference,
                candidate
            )
        )

    if not candidates:
        return None

    # First match historical frequency,
    # then recency
    candidates.sort(
        key=lambda x: (
            x[0],
            x[1]
        )
    )

    return candidates[0][2]

In [69]:
matched_events = 0

for _, row in test_edges.iterrows():

    negative = choose_matched_negative(
        row["source"],
        row["target"],
        row["timestamp"]
    )

    if negative is not None:
        matched_events += 1

print(
    "Matched hard-negative events:",
    matched_events
)

print(
    "Percentage of test:",
    matched_events / len(test_edges) * 100,
    "%"
)

Matched hard-negative events: 11801
Percentage of test: 37.90389927410548 %


In [70]:
from sklearn.metrics import roc_auc_score, average_precision_score

temporal_model.eval()
link_predictor.eval()

memory_eval = memory.detach().clone()

matched_y_true = []
matched_y_scores = []

matched_frequency_scores = []

rng = np.random.default_rng(42)

evaluated_events = 0

with torch.no_grad():

    for i in range(test_edge_index.shape[1]):

        source = test_edge_index[0, i]
        target = test_edge_index[1, i]

        source_id = source.item()
        target_id = target.item()

        source_wallet = id_to_wallet[source_id]
        target_wallet = id_to_wallet[target_id]

        test_time = test_edges.iloc[i]["timestamp"]

        # --------------------------------
        # Find matched historical negative
        # --------------------------------

        negative_wallet = choose_matched_negative(
            source_wallet,
            target_wallet,
            test_time
        )

        # No valid matched negative
        if negative_wallet is None:

            # Still reveal actual event
            memory_eval = temporal_model(
                memory_eval,
                source.unsqueeze(0),
                target.unsqueeze(0),
                test_edge_attr[i].unsqueeze(0)
            )

            continue

        negative_id = wallet_to_id[
            negative_wallet
        ]

        negative_target = torch.tensor(
            [negative_id],
            dtype=torch.long,
            device=device
        )

        # --------------------------------
        # TGNN positive
        # --------------------------------

        source_memory = memory_eval[source]
        target_memory = memory_eval[target]

        positive_score = link_predictor(
            source_memory.unsqueeze(0),
            target_memory.unsqueeze(0)
        ).item()

        # --------------------------------
        # TGNN matched negative
        # --------------------------------

        negative_memory = memory_eval[
            negative_target
        ]

        negative_score = link_predictor(
            source_memory.unsqueeze(0),
            negative_memory
        ).item()

        matched_y_true.extend([1, 0])

        matched_y_scores.extend([
            positive_score,
            negative_score
        ])

        # --------------------------------
        # Frequency baseline
        # --------------------------------

        positive_frequency = pair_counts.get(
            (source_wallet, target_wallet),
            0
        )

        negative_frequency = pair_counts.get(
            (source_wallet, negative_wallet),
            0
        )

        matched_frequency_scores.extend([
            positive_frequency,
            negative_frequency
        ])

        evaluated_events += 1

        # --------------------------------
        # Reveal actual future event
        # --------------------------------

        memory_eval = temporal_model(
            memory_eval,
            source.unsqueeze(0),
            target.unsqueeze(0),
            test_edge_attr[i].unsqueeze(0)
        )


# --------------------------------
# Metrics
# --------------------------------

matched_tgnn_auc = roc_auc_score(
    matched_y_true,
    matched_y_scores
)

matched_tgnn_ap = average_precision_score(
    matched_y_true,
    matched_y_scores
)

matched_freq_auc = roc_auc_score(
    matched_y_true,
    matched_frequency_scores
)

matched_freq_ap = average_precision_score(
    matched_y_true,
    matched_frequency_scores
)

print("Matched hard-negative evaluation")
print("---------------------------------")
print("Events evaluated:", evaluated_events)

print()
print("TGNN")
print(f"ROC-AUC: {matched_tgnn_auc:.4f}")
print(f"Average Precision: {matched_tgnn_ap:.4f}")

print()
print("Historical Frequency")
print(f"ROC-AUC: {matched_freq_auc:.4f}")
print(f"Average Precision: {matched_freq_ap:.4f}")

Matched hard-negative evaluation
---------------------------------
Events evaluated: 11801

TGNN
ROC-AUC: 0.6305
Average Precision: 0.5547

Historical Frequency
ROC-AUC: 0.5107
Average Precision: 0.5189


The TGNN's very high performance on random negatives is partly due to the relative ease of distinguishing actual interacting pairs from arbitrary wallet pairs. When negatives are matched on historical interaction frequency and recency, performance decreases substantially, but remains above chance.

In [71]:
from collections import defaultdict

# Historical neighbors for each source
historical_neighbors = defaultdict(set)

for _, row in train_edges.iterrows():

    historical_neighbors[
        row["source"]
    ].add(
        row["target"]
    )

# Only evaluate events where the source
# has at least 5 historical neighbors.
#
# This gives us meaningful candidate sets
# instead of ranking among 1-2 wallets.

ranking_events = []

for _, row in test_edges.iterrows():

    source = row["source"]
    target = row["target"]

    candidates = set(
        historical_neighbors[source]
    )

    candidates.discard(source)

    # Actual future target must be included
    candidates.add(target)

    if len(candidates) >= 5:

        ranking_events.append({
            "source": source,
            "target": target,
            "timestamp": row["timestamp"],
            "candidates": list(candidates)
        })

print(
    "Ranking events:",
    len(ranking_events)
)

print(
    "Percentage of test:",
    len(ranking_events) / len(test_edges) * 100,
    "%"
)

Ranking events: 25134
Percentage of test: 80.72846405858547 %


In [72]:
temporal_model.eval()
link_predictor.eval()

memory_eval = memory.detach().clone()

reciprocal_ranks = []

hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0

evaluated_events = 0

with torch.no_grad():

    ranking_lookup = {
        (
            event["source"],
            event["target"],
            event["timestamp"]
        ): event
        for event in ranking_events
    }

    for i in range(test_edge_index.shape[1]):

        source = test_edge_index[0, i]
        target = test_edge_index[1, i]

        source_wallet = id_to_wallet[
            source.item()
        ]

        target_wallet = id_to_wallet[
            target.item()
        ]

        test_time = test_edges.iloc[i]["timestamp"]

        key = (
            source_wallet,
            target_wallet,
            test_time
        )

        event = ranking_lookup.get(key)

        # --------------------------------
        # Skip events without enough candidates
        # --------------------------------

        if event is None:

            memory_eval = temporal_model(
                memory_eval,
                source.unsqueeze(0),
                target.unsqueeze(0),
                test_edge_attr[i].unsqueeze(0)
            )

            continue

        candidates = event["candidates"]

        # --------------------------------
        # Source memory
        # --------------------------------

        source_memory = memory_eval[source]

        candidate_ids = torch.tensor(
            [
                wallet_to_id[c]
                for c in candidates
            ],
            dtype=torch.long,
            device=device
        )

        candidate_memory = memory_eval[
            candidate_ids
        ]

        # Repeat source memory for every candidate
        source_repeated = source_memory.unsqueeze(0).repeat(
            len(candidates),
            1
        )

        # --------------------------------
        # Score all candidates
        # --------------------------------

        scores = link_predictor(
            source_repeated,
            candidate_memory
        )

        scores = scores.cpu().numpy()

        # Higher score = more likely
        ranked_indices = np.argsort(
            -scores
        )

        ranked_candidates = [
            candidates[j]
            for j in ranked_indices
        ]

        # Position of true target
        rank = (
            ranked_candidates.index(
                target_wallet
            ) + 1
        )

        reciprocal_ranks.append(
            1.0 / rank
        )

        if rank <= 1:
            hits_at_1 += 1

        if rank <= 5:
            hits_at_5 += 1

        if rank <= 10:
            hits_at_10 += 1

        evaluated_events += 1

        # --------------------------------
        # Reveal actual event
        # --------------------------------

        memory_eval = temporal_model(
            memory_eval,
            source.unsqueeze(0),
            target.unsqueeze(0),
            test_edge_attr[i].unsqueeze(0)
        )


# --------------------------------
# Final metrics
# --------------------------------

mrr = np.mean(reciprocal_ranks)

hits1 = hits_at_1 / evaluated_events
hits5 = hits_at_5 / evaluated_events
hits10 = hits_at_10 / evaluated_events

print("Temporal ranking evaluation")
print("----------------------------")
print(f"Events evaluated: {evaluated_events:,}")
print()
print(f"MRR:     {mrr:.4f}")
print(f"Hits@1:  {hits1:.4f}")
print(f"Hits@5:  {hits5:.4f}")
print(f"Hits@10: {hits10:.4f}")

Temporal ranking evaluation
----------------------------
Events evaluated: 25,134

MRR:     0.1341
Hits@1:  0.0285
Hits@5:  0.2050
Hits@10: 0.3480


In [73]:
frequency_reciprocal_ranks = []

frequency_hits_at_1 = 0
frequency_hits_at_5 = 0
frequency_hits_at_10 = 0

for event in ranking_events:

    source = event["source"]
    target = event["target"]
    candidates = event["candidates"]

    # Score candidates using ONLY training frequency
    scores = np.array([
        pair_counts.get(
            (source, candidate),
            0
        )
        for candidate in candidates
    ])

    ranked_indices = np.argsort(
        -scores
    )

    ranked_candidates = [
        candidates[j]
        for j in ranked_indices
    ]

    rank = (
        ranked_candidates.index(target)
        + 1
    )

    frequency_reciprocal_ranks.append(
        1.0 / rank
    )

    if rank <= 1:
        frequency_hits_at_1 += 1

    if rank <= 5:
        frequency_hits_at_5 += 1

    if rank <= 10:
        frequency_hits_at_10 += 1


frequency_mrr = np.mean(
    frequency_reciprocal_ranks
)

frequency_hits1 = (
    frequency_hits_at_1 /
    len(ranking_events)
)

frequency_hits5 = (
    frequency_hits_at_5 /
    len(ranking_events)
)

frequency_hits10 = (
    frequency_hits_at_10 /
    len(ranking_events)
)

print("Historical frequency ranking")
print("----------------------------")
print(f"MRR:     {frequency_mrr:.4f}")
print(f"Hits@1:  {frequency_hits1:.4f}")
print(f"Hits@5:  {frequency_hits5:.4f}")
print(f"Hits@10: {frequency_hits10:.4f}")

Historical frequency ranking
----------------------------
MRR:     0.1162
Hits@1:  0.0434
Hits@5:  0.1574
Hits@10: 0.2639


In [99]:
temporal_model.eval()
link_predictor.eval()

with torch.no_grad():
    memory = initial_memory.clone()   # same feature-based, normalized init used in training
    
    chunk_size = 2000
    for start in range(0, train_edge_index.shape[1], chunk_size):
        end = min(start + chunk_size, train_edge_index.shape[1])
        for i in range(start, end):
            source = train_edge_index[0, i].unsqueeze(0)
            target = train_edge_index[1, i].unsqueeze(0)
            memory = temporal_model(memory, source, target, train_edge_attr[i].unsqueeze(0))

tgnn_embeddings = memory.cpu().numpy()

print("TGNN embeddings shape:", tgnn_embeddings.shape)

TGNN embeddings shape: (2356, 32)


In [100]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

results = []

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(tgnn_embeddings)

    silhouette = silhouette_score(
        tgnn_embeddings,
        labels
    )

    results.append({
        "k": k,
        "silhouette": silhouette
    })

    print(
        f"K={k}: silhouette={silhouette:.4f}"
    )

K=2: silhouette=0.6069
K=3: silhouette=0.5699
K=4: silhouette=0.5781
K=5: silhouette=0.5782
K=6: silhouette=0.5175
K=7: silhouette=0.5246
K=8: silhouette=0.5313
K=9: silhouette=0.5329
K=10: silhouette=0.5373


In [101]:
results_df = pd.DataFrame(results)

best_k = results_df.loc[
    results_df["silhouette"].idxmax(),
    "k"
]

print("Best K:", best_k)

tgnn_kmeans = KMeans(
    n_clusters=int(best_k),
    random_state=42,
    n_init=20
)

tgnn_labels = tgnn_kmeans.fit_predict(
    tgnn_embeddings
)

print(
    "Best silhouette:",
    results_df["silhouette"].max()
)

Best K: 2
Best silhouette: 0.6069052815437317


In [102]:
cluster_analysis = node_features_graph.copy()

cluster_analysis["cluster"] = tgnn_labels

summary = (
    cluster_analysis
    .groupby("cluster")
    .agg(
        wallets=("proxyWallet", "count"),
        mean_trades=("trade_count", "mean"),
        median_trades=("trade_count", "median"),
        mean_size=("avg_trade_size", "mean"),
        mean_price=("avg_price", "mean"),
        buy_ratio=("buy_ratio", "mean"),
        sell_ratio=("sell_ratio", "mean"),
        mean_duration=("active_duration_hours", "mean"),
    )
    .round(3)
)

summary

,wallets,mean_trades,median_trades,mean_size,mean_price,buy_ratio,sell_ratio,mean_duration
cluster,,,,,,,,
0,2007,4.768,2.0,781.984,0.715,0.415,0.585,161.516
1,349,1.232,1.0,313.272,0.688,0.968,0.032,2.700


In [103]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Feature-only clustering
feature_labels = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=20
).fit_predict(
    node_features_graph[
        [
            "log_trade_count",
            "buy_ratio",
            "sell_ratio",
            "log_avg_trade_size",
            "log_median_trade_size",
            "avg_price",
            "price_std",
            "log_active_duration"
        ]
    ]
)

# GraphSAGE clustering
graphsage_labels = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
).fit_predict(
    graphsage_embeddings
)

# TGNN clustering
tgnn_labels_3 = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
).fit_predict(
    tgnn_embeddings
)

print("TGNN vs feature-only")
print(
    "ARI:",
    adjusted_rand_score(
        feature_labels,
        tgnn_labels_3
    )
)

print(
    "NMI:",
    normalized_mutual_info_score(
        feature_labels,
        tgnn_labels_3
    )
)

print("\nTGNN vs GraphSAGE")
print(
    "ARI:",
    adjusted_rand_score(
        graphsage_labels,
        tgnn_labels_3
    )
)

print(
    "NMI:",
    normalized_mutual_info_score(
        graphsage_labels,
        tgnn_labels_3
    )
)

NameError: name 'graphsage_embeddings' is not defined

In [104]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Feature-only clustering
feature_columns = [
    "log_trade_count",
    "buy_ratio",
    "sell_ratio",
    "log_avg_trade_size",
    "log_median_trade_size",
    "avg_price",
    "price_std",
    "log_active_duration"
]

feature_labels = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=20
).fit_predict(
    node_features_graph[feature_columns]
)

# TGNN clustering with K=3
tgnn_labels_3 = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
).fit_predict(
    tgnn_embeddings
)

print("TGNN vs feature-only clustering")
print("--------------------------------")
print(
    "ARI:",
    adjusted_rand_score(
        feature_labels,
        tgnn_labels_3
    )
)

print(
    "NMI:",
    normalized_mutual_info_score(
        feature_labels,
        tgnn_labels_3
    )
)

TGNN vs feature-only clustering
--------------------------------
ARI: 0.020736052176087667
NMI: 0.10273580953046924


In [105]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Cosine similarity between all wallet embeddings
similarity_matrix = cosine_similarity(tgnn_embeddings)

# Build actual interaction counts
interaction_counts = (
    temporal_edges
    .groupby(["source", "target"])
    .size()
    .to_dict()
)

# Collect wallet pairs
pair_results = []

num_nodes = len(id_to_wallet)

for i in range(num_nodes):

    for j in range(i + 1, num_nodes):

        wallet_i = id_to_wallet[i]
        wallet_j = id_to_wallet[j]

        similarity = similarity_matrix[i, j]

        interactions = (
            interaction_counts.get(
                (wallet_i, wallet_j),
                0
            )
            +
            interaction_counts.get(
                (wallet_j, wallet_i),
                0
            )
        )

        pair_results.append({
            "wallet_i": wallet_i,
            "wallet_j": wallet_j,
            "similarity": similarity,
            "interactions": interactions
        })

pair_df = pd.DataFrame(pair_results)

print(pair_df.shape)
pair_df.head()

(2774190, 4)


,wallet_i,wallet_j,similarity,interactions
0,0x00039cdf685d036595ed8f079a897de78f75e331,0x0072e290392e1165247c8c2ade58f043fe8a191e,0.922785,0
1,0x00039cdf685d036595ed8f079a897de78f75e331,0x0086646c84c241f443a9d51ee594e9d17d3677cb,0.790895,0
2,0x00039cdf685d036595ed8f079a897de78f75e331,0x00a7af94d4e0316cce950e80652e1eaa6f00f430,0.975413,0
3,0x00039cdf685d036595ed8f079a897de78f75e331,0x00fea6a543b001c243afadeb90edfd416a11abeb,0.398051,0
4,0x00039cdf685d036595ed8f079a897de78f75e331,0x015b5f2bf3949b2f9d80518d73cbe09d0567b93b,0.929291,0


In [81]:
from scipy.stats import spearmanr

# Use log interaction count because interaction counts are highly skewed
pair_df["log_interactions"] = np.log1p(
    pair_df["interactions"]
)

rho, p_value = spearmanr(
    pair_df["similarity"],
    pair_df["log_interactions"]
)

print("TGNN similarity vs interaction frequency")
print("-----------------------------------------")
print("Spearman rho:", rho)
print("p-value:", p_value)

TGNN similarity vs interaction frequency
-----------------------------------------
Spearman rho: 0.09379203586959
p-value: 0.0


In [82]:
# Get wallet embeddings at the end of the training period
# No test events are processed here.

temporal_model.eval()

with torch.no_grad():

    train_memory = torch.zeros(
        data.x.shape[0],
        32,
        device=device
    )

    chunk_size = 2000

    for start in range(
        0,
        train_edge_index.shape[1],
        chunk_size
    ):

        end = min(
            start + chunk_size,
            train_edge_index.shape[1]
        )

        for i in range(start, end):

            source = train_edge_index[0, i].unsqueeze(0)
            target = train_edge_index[1, i].unsqueeze(0)

            train_memory = temporal_model(
                train_memory,
                source,
                target,
                train_edge_attr[i].unsqueeze(0)
            )

        print(
            f"Processed {end:,} / "
            f"{train_edge_index.shape[1]:,}"
        )

train_embeddings = train_memory.cpu().numpy()

print()
print("Train-boundary embeddings:", train_embeddings.shape)

Processed 2,000 / 72,645
Processed 4,000 / 72,645
Processed 6,000 / 72,645
Processed 8,000 / 72,645
Processed 10,000 / 72,645
Processed 12,000 / 72,645
Processed 14,000 / 72,645
Processed 16,000 / 72,645
Processed 18,000 / 72,645
Processed 20,000 / 72,645
Processed 22,000 / 72,645
Processed 24,000 / 72,645
Processed 26,000 / 72,645
Processed 28,000 / 72,645
Processed 30,000 / 72,645
Processed 32,000 / 72,645
Processed 34,000 / 72,645
Processed 36,000 / 72,645
Processed 38,000 / 72,645
Processed 40,000 / 72,645
Processed 42,000 / 72,645
Processed 44,000 / 72,645
Processed 46,000 / 72,645
Processed 48,000 / 72,645
Processed 50,000 / 72,645
Processed 52,000 / 72,645
Processed 54,000 / 72,645
Processed 56,000 / 72,645
Processed 58,000 / 72,645
Processed 60,000 / 72,645
Processed 62,000 / 72,645
Processed 64,000 / 72,645
Processed 66,000 / 72,645
Processed 68,000 / 72,645
Processed 70,000 / 72,645
Processed 72,000 / 72,645
Processed 72,645 / 72,645

Train-boundary embeddings: (2356, 32)


In [83]:
# Unique wallet pairs that interact during the unseen test period

test_pairs = (
    test_edges[
        ["source", "target"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

test_pairs["source_id"] = test_pairs[
    "source"
].map(wallet_to_id)

test_pairs["target_id"] = test_pairs[
    "target"
].map(wallet_to_id)

test_pairs = test_pairs.dropna().reset_index(drop=True)

print("Unique future interaction pairs:", len(test_pairs))

Unique future interaction pairs: 9480


In [84]:
from sklearn.metrics.pairwise import cosine_similarity

positive_source = test_pairs["source_id"].astype(int).values
positive_target = test_pairs["target_id"].astype(int).values

positive_similarity = np.sum(
    train_embeddings[positive_source] *
    train_embeddings[positive_target],
    axis=1
)

positive_norms = (
    np.linalg.norm(
        train_embeddings[positive_source],
        axis=1
    )
    *
    np.linalg.norm(
        train_embeddings[positive_target],
        axis=1
    )
)

positive_similarity = (
    positive_similarity /
    (positive_norms + 1e-8)
)

print(
    "Positive pairs:",
    len(positive_similarity)
)

print(
    "Mean similarity:",
    positive_similarity.mean()
)

print(
    "Median similarity:",
    np.median(positive_similarity)
)

Positive pairs: 9480
Mean similarity: 0.5632049
Median similarity: 0.9998936


In [85]:
print("Embedding shape:", train_embeddings.shape)

# Basic statistics
print("\nEmbedding statistics")
print("--------------------")
print("Mean:", train_embeddings.mean())
print("Std:", train_embeddings.std())
print("Min:", train_embeddings.min())
print("Max:", train_embeddings.max())

# Embedding norms
norms = np.linalg.norm(
    train_embeddings,
    axis=1
)

print("\nEmbedding norms")
print("----------------")
print("Mean:", norms.mean())
print("Std:", norms.std())
print("Min:", norms.min())
print("Max:", norms.max())

# Pairwise cosine similarity sample
from sklearn.metrics.pairwise import cosine_similarity

sample_size = min(500, len(train_embeddings))

sample_embeddings = train_embeddings[:sample_size]

sim = cosine_similarity(sample_embeddings)

# Ignore diagonal
upper = sim[
    np.triu_indices(sample_size, k=1)
]

print("\nRandom pair cosine similarity")
print("-----------------------------")
print("Mean:", upper.mean())
print("Median:", np.median(upper))
print("Std:", upper.std())
print("Min:", upper.min())
print("Max:", upper.max())

Embedding shape: (2356, 32)

Embedding statistics
--------------------
Mean: 0.21204391
Std: 0.7987049
Min: -0.9997654
Max: 0.9997104

Embedding norms
----------------
Mean: 4.349501
Std: 1.7130054
Min: 0.0
Max: 5.467213

Random pair cosine similarity
-----------------------------
Mean: 0.7999516
Median: 0.98829734
Std: 0.38358918
Min: 0.0
Max: 1.0000002


In [86]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import torch

temporal_model.eval()
link_predictor.eval()

# ---------------------------------------------------------
# 1. Build historical neighbors from TRAIN ONLY
# ---------------------------------------------------------

historical_neighbors = {}

for _, row in train_edges.iterrows():

    source = row["source"]
    target = row["target"]

    if source not in historical_neighbors:
        historical_neighbors[source] = set()

    historical_neighbors[source].add(target)


# ---------------------------------------------------------
# 2. Build future pairs from TEST
# ---------------------------------------------------------

future_pairs = (
    test_edges[
        ["source", "target"]
    ]
    .drop_duplicates()
)

# ---------------------------------------------------------
# 3. Score positive and hard-negative pairs
# ---------------------------------------------------------

positive_scores = []
negative_scores = []

with torch.no_grad():

    for _, row in future_pairs.iterrows():

        source_wallet = row["source"]
        target_wallet = row["target"]

        # Source must have historical neighbors
        if source_wallet not in historical_neighbors:
            continue

        candidates = list(
            historical_neighbors[source_wallet]
        )

        # Remove the actual future target
        negative_candidates = [
            c for c in candidates
            if c != target_wallet
        ]

        if not negative_candidates:
            continue

        # Pick one random historical neighbor
        negative_target = np.random.choice(
            negative_candidates
        )

        source_id = wallet_to_id[source_wallet]
        target_id = wallet_to_id[target_wallet]
        negative_id = wallet_to_id[negative_target]

        source_tensor = torch.tensor(
            [source_id],
            dtype=torch.long,
            device=device
        )

        target_tensor = torch.tensor(
            [target_id],
            dtype=torch.long,
            device=device
        )

        negative_tensor = torch.tensor(
            [negative_id],
            dtype=torch.long,
            device=device
        )

        source_memory = train_memory[
            source_tensor
        ]

        target_memory = train_memory[
            target_tensor
        ]

        negative_memory = train_memory[
            negative_tensor
        ]

        positive_score = link_predictor(
            source_memory,
            target_memory
        ).item()

        negative_score = link_predictor(
            source_memory,
            negative_memory
        ).item()

        positive_scores.append(
            positive_score
        )

        negative_scores.append(
            negative_score
        )


# ---------------------------------------------------------
# 4. Calculate metrics
# ---------------------------------------------------------

y_true = np.concatenate([
    np.ones(len(positive_scores)),
    np.zeros(len(negative_scores))
])

y_scores = np.concatenate([
    np.array(positive_scores),
    np.array(negative_scores)
])

auc = roc_auc_score(
    y_true,
    y_scores
)

ap = average_precision_score(
    y_true,
    y_scores
)

print("Out-of-time hard-negative evaluation")
print("-------------------------------------")
print(
    "Events evaluated:",
    len(positive_scores)
)

print(
    f"ROC-AUC: {auc:.4f}"
)

print(
    f"Average Precision: {ap:.4f}"
)

Out-of-time hard-negative evaluation
-------------------------------------
Events evaluated: 6960
ROC-AUC: 0.3707
Average Precision: 0.4402


In [87]:
# How much of the embedding space is 1-dimensional in practice?
from numpy.linalg import svd

centered = tgnn_embeddings - tgnn_embeddings.mean(axis=0)
u, s, vt = svd(centered, full_matrices=False)
explained_var_ratio = (s**2) / (s**2).sum()
print("Top 5 components explain:", explained_var_ratio[:5].cumsum())

Top 5 components explain: [0.96711344 0.9974024  0.9985347  0.9994041  0.9997753 ]


In [88]:
pc1 = u[:, 0] * s[0]  # projection onto first principal component

from scipy.stats import pearsonr
print("PC1 vs trade_count:", pearsonr(pc1, wallet_features["trade_count"])[0])
print("PC1 vs active_duration:", pearsonr(pc1, wallet_features["active_duration_hours"])[0])

PC1 vs trade_count: -0.07590579662242132
PC1 vs active_duration: -0.2285054871145675


In [89]:
# 1. Was the collapse already present at initialization, before any training?
init_centered = initial_memory.detach().cpu().numpy()
init_centered = init_centered - init_centered.mean(axis=0)
u0, s0, vt0 = np.linalg.svd(init_centered, full_matrices=False)
print("Initial (untrained) PC1 share:", (s0[0]**2) / (s0**2).sum())

Initial (untrained) PC1 share: nan


/var/folders/52/5f__721d2gsbnkm_q6cvchyh0000gn/T/ipykernel_63142/1166668582.py:5: RuntimeWarning: invalid value encountered in scalar divide
  print("Initial (untrained) PC1 share:", (s0[0]**2) / (s0**2).sum())


In [90]:
# 2. Correlate PC1 with every individual node feature column
feature_names = ["log_trade_count", "buy_ratio", "sell_ratio", "log_avg_trade_size",
                  "log_median_trade_size", "avg_price", "price_std", "log_active_duration",
                  "<9th column>"]  # adjust to match your actual 9 x-columns

x_np = data.x.cpu().numpy()
for k in range(x_np.shape[1]):
    r, _ = pearsonr(pc1, x_np[:, k])
    print(f"PC1 vs feature[{k}]: {r:.4f}")

PC1 vs feature[0]: -0.3005
PC1 vs feature[1]: 0.0968
PC1 vs feature[2]: -0.0968
PC1 vs feature[3]: -0.0319
PC1 vs feature[4]: 0.0029
PC1 vs feature[5]: -0.0648
PC1 vs feature[6]: -0.1114
PC1 vs feature[7]: -0.0955
PC1 vs feature[8]: -0.3698


In [91]:
# 3. Correlate PC1 with train-graph degree (not trade_count — actual node degree in edge_index)
train_degree = torch.bincount(train_edge_index[0].cpu(), minlength=num_nodes) + \
               torch.bincount(train_edge_index[1].cpu(), minlength=num_nodes)
train_degree = train_degree.numpy()

r, _ = pearsonr(pc1, train_degree)
print("PC1 vs train graph degree:", r)

# also split by isolated vs connected
print("Mean PC1, isolated nodes:", pc1[train_degree == 0].mean())
print("Mean PC1, connected nodes:", pc1[train_degree > 0].mean())

PC1 vs train graph degree: -0.09853821390819797
Mean PC1, isolated nodes: 4.287678
Mean PC1, connected nodes: -0.43277362


In [92]:
isolated_count = (train_degree == 0).sum()
print(f"Isolated wallets in train graph: {isolated_count} / {num_nodes} "
      f"({isolated_count/num_nodes*100:.1f}%)")

Isolated wallets in train graph: 216 / 2356 (9.2%)


In [93]:
class TemporalMemory(nn.Module):

    def __init__(self, node_dim, edge_dim, hidden_dim=32):
        super().__init__()
        self.message_mlp = nn.Sequential(
            nn.Linear(node_dim * 2 + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.memory_update = nn.GRUCell(hidden_dim, hidden_dim)

    def forward(self, memory, source, target, edge_attr):
        source_memory = memory[source]
        target_memory = memory[target]

        message_input = torch.cat([source_memory, target_memory, edge_attr], dim=1)
        message = self.message_mlp(message_input)

        new_source_memory = self.memory_update(message, source_memory)
        new_target_memory = self.memory_update(message, target_memory)

        # Fix 2: bound magnitude so direction has to carry the signal
        new_source_memory = F.normalize(new_source_memory, p=2, dim=-1)
        new_target_memory = F.normalize(new_target_memory, p=2, dim=-1)

        memory = memory.clone()
        memory[source] = new_source_memory
        memory[target] = new_target_memory
        return memory


class TemporalLinkPredictor(nn.Module):

    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, source_memory, target_memory):
        cos_sim = F.cosine_similarity(source_memory, target_memory, dim=-1)
        return cos_sim / self.temperature

In [94]:
# Reset everything
node_encoder = nn.Linear(x.shape[1], hidden_dim).to(device)
temporal_model = TemporalMemory(node_dim=hidden_dim, edge_dim=4, hidden_dim=hidden_dim).to(device)
link_predictor = TemporalLinkPredictor().to(device)

optimizer = torch.optim.Adam(
    list(node_encoder.parameters()) +
    list(temporal_model.parameters()) +
    list(link_predictor.parameters()),
    lr=0.001,
    weight_decay=1e-5
)

# Fix 1: real feature-based init, not zeros — also normalize so it's on the
# same unit-norm footing the GRU will maintain thereafter
initial_memory = F.normalize(node_encoder(data.x.to(device)), p=2, dim=-1)
memory = initial_memory.clone()

In [95]:
# 1. Full-population SVD
centered = tgnn_embeddings - tgnn_embeddings.mean(axis=0)
u, s, vt = np.linalg.svd(centered, full_matrices=False)
explained_var_ratio = (s**2) / (s**2).sum()
print("Full population — top 5 components explain:", explained_var_ratio[:5].cumsum())

Full population — top 5 components explain: [0.96711344 0.9974024  0.9985347  0.9994041  0.9997753 ]


In [96]:
# 2. Connected-only SVD
connected_mask = train_degree > 0
connected_embeddings = tgnn_embeddings[connected_mask]

centered_c = connected_embeddings - connected_embeddings.mean(axis=0)
u_c, s_c, vt_c = np.linalg.svd(centered_c, full_matrices=False)
explained_var_ratio_c = (s_c**2) / (s_c**2).sum()
print("Connected-only — top 5 components explain:", explained_var_ratio_c[:5].cumsum())

Connected-only — top 5 components explain: [0.9866305  0.99559355 0.9978145  0.99914753 0.9995014 ]


In [97]:
# 3. Isolated vs connected — are they still collapsing to one point?
print("Isolated wallets — embedding std across wallets:", tgnn_embeddings[~connected_mask].std(axis=0).mean())
print("Connected wallets — embedding std across wallets:", tgnn_embeddings[connected_mask].std(axis=0).mean())

# And the random-pair cosine similarity check from before, re-run on the new embeddings
from sklearn.metrics.pairwise import cosine_similarity
sample_size = min(500, len(tgnn_embeddings))
sim = cosine_similarity(tgnn_embeddings[:sample_size])
upper = sim[np.triu_indices(sample_size, k=1)]
print("\nRandom pair cosine similarity — mean:", upper.mean(), "median:", np.median(upper))

Isolated wallets — embedding std across wallets: 0.0
Connected wallets — embedding std across wallets: 0.1997933

Random pair cosine similarity — mean: 0.7999516 median: 0.98829734


In [98]:
print("Isolated wallet embedding std:", tgnn_embeddings[~connected_mask].std(axis=0).mean())
print("PC1 share:", explained_var_ratio[0])

Isolated wallet embedding std: 0.0
PC1 share: 0.96711344
